## ## Add Merged Cluster Labels and Sample Reference h5ad Files

This notebook processes reference .h5ad files, such as TSP-HBA and TSP-BDa, by mapping cell_ontology_class annotations to merged cell-type clusters. For each mapped cell type, the notebook samples up to 1500 cells, or retains all available cells if fewer than 1500 are present. Cells without a corresponding merged cluster assignment are removed. The two pancreatic pp cells are also removed, as they are either filtered out by the other algorithms or intentionally removed for BayesPrism. A new obs column, MergedClusters, is then added to store the merged cluster label for each retained cell.

The resulting downsampled and annotated AnnData object is saved as a new `.h5ad` file for downstream DECODE analyses.

In [1]:
import scanpy as sc
import pandas as pd

h5ad_path = "/gpfs/igmmfs01/datastore/wendy-lab/Alexis/CDA-Data/03_TSP_Matrix-Generation/HumanBrainAtlas-snRNA-V1/TSP-HBA-QC-Inner.h5ad"
clusters_path = "/gpfs/igmmfs01/datastore/wendy-lab/Alexis/CDA-Data/03_TSP_Matrix-Generation/TSP-HBA_Inner_All-Compartments_clusters.csv"

cell_col = "cell_ontology_class"
new_col = "MergedClusters"

# Load h5ad
adata = sc.read_h5ad(h5ad_path)

# Load merged cluster table
clusters_df = pd.read_csv(clusters_path, index_col=0)

# Build mapping: individual cell type -> merged cluster name
category_mapping = {}

for merged_category in clusters_df.index:
    individual_types = str(merged_category).split("/")
    for cell_type in individual_types:
        category_mapping[cell_type.strip()] = merged_category

# Map cell type to merged cluster
mapped = adata.obs[cell_col].astype(str).map(category_mapping)

# Report unmapped before removing
print(f"Cells before filtering: {adata.n_obs:,}")
print(f"Unmapped cells to remove: {mapped.isna().sum():,}")

print("\nUnmapped cell types:")
print(
    adata.obs.loc[mapped.isna(), cell_col]
    .astype(str)
    .value_counts()
)

# Keep only mapped cells
adata = adata[mapped.notna()].copy()

# Add MergedClusters column
adata.obs[new_col] = (
    adata.obs[cell_col]
    .astype(str)
    .map(category_mapping)
)

print(f"\nCells after filtering: {adata.n_obs:,}")
print(f"Number of merged clusters: {adata.obs[new_col].nunique()}")

print("\nMergedClusters counts:")
print(adata.obs[new_col].value_counts())

out_path = "/gpfs/igmmfs01/datastore/wendy-lab/Alexis/CDA-Data/03_TSP_Matrix-Generation/HumanBrainAtlas-snRNA-V1/TSP-HBA-QC-Inner_MergedClusters_DECODE.h5ad"
adata.write_h5ad(out_path)

print(f"Saved: {out_path}")

Cells before filtering: 602,395
Unmapped cells to remove: 22,409

Unmapped cell types:
cell_ontology_class
skeletal muscle satellite stem cell       4131
conjunctival epithelial cell              3799
myofibroblast cell                        2679
thymic fibroblast type 1                  2085
fibroblast of breast                      1957
corneal epithelial cell                   1495
thymic fibroblast type 2                  1296
eye photoreceptor cell                     871
hematopoietic precursor cell               867
vascular associated smooth muscle cell     630
keratocyte                                 577
retinal blood vessel endothelial cell      513
mueller cell                               338
alveolar type 2 fibroblast cell            226
fibroblast of cardiac tissue               209
microglial cell                            155
connective tissue cell                     124
acinar cell                                109
bronchial smooth muscle cell                79


In [3]:
from random import sample, seed
import scanpy as sc
import pandas as pd
from scipy.sparse import issparse

SEED = 42
seed(SEED)

h5ad_path = "/gpfs/igmmfs01/datastore/wendy-lab/Alexis/CDA-Data/03_TSP_Matrix-Generation/HumanBrainAtlas-snRNA-V1/TSP-HBA-QC-Inner.h5ad"
clusters_path = "/gpfs/igmmfs01/datastore/wendy-lab/Alexis/CDA-Data/03_TSP_Matrix-Generation/TSP-HBA_Inner_All-Compartments_clusters.csv"

cellAnnotCol = "cell_ontology_class"
new_col = "MergedClusters"
maxObs = 1500

adata = sc.read_h5ad(h5ad_path)

# Rename annotation before matching to cluster definitions
adata.obs[cellAnnotCol] = (adata.obs[cellAnnotCol].replace({"endothelial": "endothelial cell"}))

if "counts" in adata.layers:
    adata.X = adata.layers["counts"].copy()

df = pd.read_csv(clusters_path, index_col=0)

goodAnnot = set()
category_mapping = {}

for merged_category in df.index:
    individual_types = str(merged_category).split("/")
    for cell_type in individual_types:
        cleaned = cell_type.strip()
        goodAnnot.add(cleaned)
        category_mapping[cleaned] = merged_category

goodAnnot = sorted(goodAnnot)

# Remove unwanted cell type
goodAnnot = [x for x in goodAnnot if x != "pancreatic pp cell"]
category_mapping.pop("pancreatic pp cell", None)

sampled_cell_ids = []

for cell in goodAnnot:
    obsThisCell = adata[adata.obs[cellAnnotCol].astype(str) == cell]
    numObs = obsThisCell.shape[0]

    print(f"Processing cell type: {cell}, found {numObs} cells")

    if numObs >= maxObs:
        indexSample = sample(range(0, numObs), maxObs)
    else:
        indexSample = range(0, numObs)

    sampled_ids = obsThisCell.obs.iloc[list(indexSample)].index.tolist()
    sampled_cell_ids.extend(sampled_ids)

print(f"Sampled cell IDs: {len(sampled_cell_ids)}")

adata_sub = adata[sampled_cell_ids].copy()

adata_sub.obs[new_col] = (
    adata_sub.obs[cellAnnotCol]
    .astype(str)
    .map(category_mapping)
)

adata_sub = adata_sub[adata_sub.obs[new_col].notna()].copy()

print(adata_sub)
print(adata_sub.obs[new_col].value_counts())

if issparse(adata_sub.X):
    adata_sub.X = adata_sub.X.toarray().astype("float32")

out_path = "/gpfs/igmmfs01/datastore/wendy-lab/Alexis/CDA-Data/03_TSP_Matrix-Generation/HumanBrainAtlas-snRNA-V1/TSP-HBA-QC-Inner_matched_DECODE_1500each.h5ad"
adata_sub.write_h5ad(out_path)

print(f"Saved: {out_path}")

Processing cell type: Bergmann glial cell, found 474 cells
Processing cell type: acinar cell of salivary gland, found 6934 cells
Processing cell type: activated cd4-positive, alpha-beta t cell, found 281 cells
Processing cell type: activated cd8-positive, alpha-beta t cell, found 6 cells
Processing cell type: adventitial cell, found 384 cells
Processing cell type: astrocyte, found 7298 cells
Processing cell type: b cell, found 37064 cells
Processing cell type: basal cell, found 10882 cells
Processing cell type: basal cell of prostate epithelium, found 2624 cells
Processing cell type: basophil, found 612 cells
Processing cell type: best4+ intestinal epithelial cell, human, found 92 cells
Processing cell type: bladder urothelial cell, found 3609 cells
Processing cell type: capillary endothelial cell, found 4473 cells
Processing cell type: cardiac endothelial cell, found 2349 cells
Processing cell type: cd4-positive, alpha-beta t cell, found 39533 cells
Processing cell type: cd4-positive,

In [3]:
import scanpy as sc
import pandas as pd

h5ad_path = "/gpfs/igmmfs01/datastore/wendy-lab/Alexis/CDA-Data/03_TSP_Matrix-Generation/DarmanisData/TSP-BDa-merged-Inner.h5ad"
clusters_path = "/gpfs/igmmfs01/datastore/wendy-lab/Alexis/CDA-Data/03_TSP_Matrix-Generation/TSP-BDa_Inner_All-Compartments_clusters.csv"

cell_col = "cell_ontology_class"
new_col = "MergedClusters"

# Load h5ad
adata = sc.read_h5ad(h5ad_path)

# Load merged cluster table
clusters_df = pd.read_csv(clusters_path, index_col=0)

# Build mapping: individual cell type -> merged cluster name
category_mapping = {}

for merged_category in clusters_df.index:
    individual_types = str(merged_category).split("/")
    for cell_type in individual_types:
        category_mapping[cell_type.strip()] = merged_category

# Map cell type to merged cluster
mapped = adata.obs[cell_col].astype(str).map(category_mapping)

# Report unmapped before removing
print(f"Cells before filtering: {adata.n_obs:,}")
print(f"Unmapped cells to remove: {mapped.isna().sum():,}")

print("\nUnmapped cell types:")
print(
    adata.obs.loc[mapped.isna(), cell_col]
    .astype(str)
    .value_counts()
)

# Keep only mapped cells
adata = adata[mapped.notna()].copy()

# Add MergedClusters column
adata.obs[new_col] = (
    adata.obs[cell_col]
    .astype(str)
    .map(category_mapping)
)

print(f"\nCells after filtering: {adata.n_obs:,}")
print(f"Number of merged clusters: {adata.obs[new_col].nunique()}")

print("\nMergedClusters counts:")
print(adata.obs[new_col].value_counts())

out_path = "/gpfs/igmmfs01/datastore/wendy-lab/Alexis/CDA-Data/03_TSP_Matrix-Generation/DarmanisData/TSP-BDa-merged-Inner_MergedClusters_DECODE.h5ad"
adata.write_h5ad(out_path)

print(f"Saved: {out_path}")

Cells before filtering: 418,522
Unmapped cells to remove: 22,395

Unmapped cell types:
cell_ontology_class
skeletal muscle satellite stem cell       4131
conjunctival epithelial cell              3799
myofibroblast cell                        2679
thymic fibroblast type 1                  2085
fibroblast of breast                      1957
corneal epithelial cell                   1495
thymic fibroblast type 2                  1296
eye photoreceptor cell                     871
hematopoietic precursor cell               867
vascular associated smooth muscle cell     597
keratocyte                                 577
retinal blood vessel endothelial cell      513
mueller cell                               338
alveolar type 2 fibroblast cell            226
fibroblast of cardiac tissue               209
microglial cell                            155
connective tissue cell                     124
acinar cell                                109
bronchial smooth muscle cell                79


In [1]:
from random import sample, seed
import scanpy as sc
import pandas as pd
from scipy.sparse import issparse

SEED = 42
seed(SEED)

h5ad_path = "/gpfs/igmmfs01/datastore/wendy-lab/Alexis/CDA-Data/03_TSP_Matrix-Generation/DarmanisData/TSP-BDa-merged-Inner.h5ad"
clusters_path = "/gpfs/igmmfs01/datastore/wendy-lab/Alexis/CDA-Data/03_TSP_Matrix-Generation/TSP-BDa_Inner_All-Compartments_clusters.csv"

cellAnnotCol = "cell_ontology_class"
new_col = "MergedClusters"
maxObs = 1500

adata = sc.read_h5ad(h5ad_path)

# Rename annotation before matching to cluster definitions
adata.obs[cellAnnotCol] = (adata.obs[cellAnnotCol].replace({"endothelial": "endothelial cell"}))

if "counts" in adata.layers:
    adata.X = adata.layers["counts"].copy()

df = pd.read_csv(clusters_path, index_col=0)

goodAnnot = set()
category_mapping = {}

for merged_category in df.index:
    individual_types = str(merged_category).split("/")
    for cell_type in individual_types:
        cleaned = cell_type.strip()
        goodAnnot.add(cleaned)
        category_mapping[cleaned] = merged_category

goodAnnot = sorted(goodAnnot)

# Remove unwanted cell type
goodAnnot = [x for x in goodAnnot if x != "pancreatic pp cell"]
category_mapping.pop("pancreatic pp cell", None)

sampled_cell_ids = []

for cell in goodAnnot:
    obsThisCell = adata[adata.obs[cellAnnotCol].astype(str) == cell]
    numObs = obsThisCell.shape[0]

    print(f"Processing cell type: {cell}, found {numObs} cells")

    if numObs >= maxObs:
        indexSample = sample(range(0, numObs), maxObs)
    else:
        indexSample = range(0, numObs)

    sampled_ids = obsThisCell.obs.iloc[list(indexSample)].index.tolist()
    sampled_cell_ids.extend(sampled_ids)

print(f"Sampled cell IDs: {len(sampled_cell_ids)}")

adata_sub = adata[sampled_cell_ids].copy()

adata_sub.obs[new_col] = (
    adata_sub.obs[cellAnnotCol]
    .astype(str)
    .map(category_mapping)
)

adata_sub = adata_sub[adata_sub.obs[new_col].notna()].copy()

print(adata_sub)
print(adata_sub.obs[new_col].value_counts())

if issparse(adata_sub.X):
    adata_sub.X = adata_sub.X.toarray().astype("float32")

out_path = "/gpfs/igmmfs01/datastore/wendy-lab/Alexis/CDA-Data/03_TSP_Matrix-Generation/DarmanisData/TSP-BDa-merged-Inner_matched_DECODE_1500each.h5ad"
adata_sub.write_h5ad(out_path)

print(f"Saved: {out_path}")

/tmp/ipykernel_3972066/3458140768.py:19: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  adata.obs[cellAnnotCol] = (adata.obs[cellAnnotCol].replace({"endothelial": "endothelial cell"}))


Processing cell type: OPC, found 18 cells
Processing cell type: acinar cell of salivary gland, found 6934 cells
Processing cell type: activated cd4-positive, alpha-beta t cell, found 281 cells
Processing cell type: activated cd8-positive, alpha-beta t cell, found 6 cells
Processing cell type: adventitial cell, found 384 cells
Processing cell type: astrocytes, found 56 cells
Processing cell type: b cell, found 37064 cells
Processing cell type: basal cell, found 10882 cells
Processing cell type: basal cell of prostate epithelium, found 2624 cells
Processing cell type: basophil, found 612 cells
Processing cell type: best4+ intestinal epithelial cell, human, found 92 cells
Processing cell type: bladder urothelial cell, found 3609 cells
Processing cell type: capillary endothelial cell, found 4473 cells
Processing cell type: cardiac endothelial cell, found 2349 cells
Processing cell type: cd4-positive, alpha-beta t cell, found 39533 cells
Processing cell type: cd4-positive, alpha-beta thymoc

In [5]:
import scanpy as sc
import pandas as pd

h5ad_path = "/gpfs/igmmfs01/datastore/wendy-lab/Alexis/CDA-Data/03_TSP_Matrix-Generation/DarmanisData/TSP-BDa-merged-Outer.h5ad"
clusters_path = "/gpfs/igmmfs01/datastore/wendy-lab/Alexis/CDA-Data/03_TSP_Matrix-Generation/TSP-BDa_Outer_All-Compartments_clusters.csv"

cell_col = "cell_ontology_class"
new_col = "MergedClusters"

# Load h5ad
adata = sc.read_h5ad(h5ad_path)

# Load merged cluster table
clusters_df = pd.read_csv(clusters_path, index_col=0)

# Build mapping: individual cell type -> merged cluster name
category_mapping = {}

for merged_category in clusters_df.index:
    individual_types = str(merged_category).split("/")
    for cell_type in individual_types:
        category_mapping[cell_type.strip()] = merged_category

# Map cell type to merged cluster
mapped = adata.obs[cell_col].astype(str).map(category_mapping)

# Report unmapped before removing
print(f"Cells before filtering: {adata.n_obs:,}")
print(f"Unmapped cells to remove: {mapped.isna().sum():,}")

print("\nUnmapped cell types:")
print(
    adata.obs.loc[mapped.isna(), cell_col]
    .astype(str)
    .value_counts()
)

# Keep only mapped cells
adata = adata[mapped.notna()].copy()

# Add MergedClusters column
adata.obs[new_col] = (
    adata.obs[cell_col]
    .astype(str)
    .map(category_mapping)
)

print(f"\nCells after filtering: {adata.n_obs:,}")
print(f"Number of merged clusters: {adata.obs[new_col].nunique()}")

print("\nMergedClusters counts:")
print(adata.obs[new_col].value_counts())

out_path = "/gpfs/igmmfs01/datastore/wendy-lab/Alexis/CDA-Data/03_TSP_Matrix-Generation/DarmanisData/TSP-BDa-merged-Outer_MergedClusters_DECODE.h5ad"
adata.write_h5ad(out_path)

print(f"Saved: {out_path}")

Cells before filtering: 418,522
Unmapped cells to remove: 22,395

Unmapped cell types:
cell_ontology_class
skeletal muscle satellite stem cell       4131
conjunctival epithelial cell              3799
myofibroblast cell                        2679
thymic fibroblast type 1                  2085
fibroblast of breast                      1957
corneal epithelial cell                   1495
thymic fibroblast type 2                  1296
eye photoreceptor cell                     871
hematopoietic precursor cell               867
vascular associated smooth muscle cell     597
keratocyte                                 577
retinal blood vessel endothelial cell      513
mueller cell                               338
alveolar type 2 fibroblast cell            226
fibroblast of cardiac tissue               209
microglial cell                            155
connective tissue cell                     124
acinar cell                                109
bronchial smooth muscle cell                79


In [2]:
from random import sample, seed
import scanpy as sc
import pandas as pd
from scipy.sparse import issparse

SEED = 42
seed(SEED)

h5ad_path = "/gpfs/igmmfs01/datastore/wendy-lab/Alexis/CDA-Data/03_TSP_Matrix-Generation/DarmanisData/TSP-BDa-merged-Outer.h5ad"
clusters_path = "/gpfs/igmmfs01/datastore/wendy-lab/Alexis/CDA-Data/03_TSP_Matrix-Generation/TSP-BDa_Outer_All-Compartments_clusters.csv"

cellAnnotCol = "cell_ontology_class"
new_col = "MergedClusters"
maxObs = 1500

adata = sc.read_h5ad(h5ad_path)

# Rename annotation before matching to cluster definitions
adata.obs[cellAnnotCol] = (adata.obs[cellAnnotCol].replace({"endothelial": "endothelial cell"}))

if "counts" in adata.layers:
    adata.X = adata.layers["counts"].copy()

df = pd.read_csv(clusters_path, index_col=0)

goodAnnot = set()
category_mapping = {}

for merged_category in df.index:
    individual_types = str(merged_category).split("/")
    for cell_type in individual_types:
        cleaned = cell_type.strip()
        goodAnnot.add(cleaned)
        category_mapping[cleaned] = merged_category

goodAnnot = sorted(goodAnnot)

# Remove unwanted cell type
goodAnnot = [x for x in goodAnnot if x != "pancreatic pp cell"]
category_mapping.pop("pancreatic pp cell", None)

sampled_cell_ids = []

for cell in goodAnnot:
    obsThisCell = adata[adata.obs[cellAnnotCol].astype(str) == cell]
    numObs = obsThisCell.shape[0]

    print(f"Processing cell type: {cell}, found {numObs} cells")

    if numObs >= maxObs:
        indexSample = sample(range(0, numObs), maxObs)
    else:
        indexSample = range(0, numObs)

    sampled_ids = obsThisCell.obs.iloc[list(indexSample)].index.tolist()
    sampled_cell_ids.extend(sampled_ids)

print(f"Sampled cell IDs: {len(sampled_cell_ids)}")

adata_sub = adata[sampled_cell_ids].copy()

adata_sub.obs[new_col] = (
    adata_sub.obs[cellAnnotCol]
    .astype(str)
    .map(category_mapping)
)

adata_sub = adata_sub[adata_sub.obs[new_col].notna()].copy()

print(adata_sub)
print(adata_sub.obs[new_col].value_counts())

if issparse(adata_sub.X):
    adata_sub.X = adata_sub.X.toarray().astype("float32")

out_path = "/gpfs/igmmfs01/datastore/wendy-lab/Alexis/CDA-Data/03_TSP_Matrix-Generation/DarmanisData/TSP-BDa-merged-Outer_matched_DECODE_1500each.h5ad"
adata_sub.write_h5ad(out_path)

print(f"Saved: {out_path}")

/tmp/ipykernel_3972066/4194444269.py:19: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  adata.obs[cellAnnotCol] = (adata.obs[cellAnnotCol].replace({"endothelial": "endothelial cell"}))


Processing cell type: OPC, found 18 cells
Processing cell type: acinar cell of salivary gland, found 6934 cells
Processing cell type: activated cd4-positive, alpha-beta t cell, found 281 cells
Processing cell type: activated cd8-positive, alpha-beta t cell, found 6 cells
Processing cell type: adventitial cell, found 384 cells
Processing cell type: astrocytes, found 56 cells
Processing cell type: b cell, found 37064 cells
Processing cell type: basal cell, found 10882 cells
Processing cell type: basal cell of prostate epithelium, found 2624 cells
Processing cell type: basophil, found 612 cells
Processing cell type: best4+ intestinal epithelial cell, human, found 92 cells
Processing cell type: bladder urothelial cell, found 3609 cells
Processing cell type: capillary endothelial cell, found 4473 cells
Processing cell type: cardiac endothelial cell, found 2349 cells
Processing cell type: cd4-positive, alpha-beta t cell, found 39533 cells
Processing cell type: cd4-positive, alpha-beta thymoc

In [7]:
import scanpy as sc
from scipy.sparse import issparse

in_path = "/gpfs/igmmfs01/datastore/wendy-lab/Alexis/CDA-Data/03_TSP_Matrix-Generation/DarmanisData/TSP-BDa-merged-Outer_matched_DECODE_1500each.h5ad"
out_path = "/gpfs/igmmfs01/datastore/wendy-lab/Alexis/CDA-Data/03_TSP_Matrix-Generation/DarmanisData/TSP-BDa-merged-Outer_matched_DECODE_1500each_20kHVG.h5ad"

adata = sc.read_h5ad(in_path)

print("Original:", adata)

# Make sure counts are used if available
if "counts" in adata.layers:
    adata.X = adata.layers["counts"].copy()

# Scanpy HVG expects normalized/log-transformed data for most flavors.
# Keep raw counts untouched in a layer.
adata.layers["counts"] = adata.X.copy()

# Normalize/log only for HVG selection
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

sc.pp.highly_variable_genes(
    adata,
    n_top_genes=20000,
    flavor="seurat",
)

adata_hvg = adata[:, adata.var["highly_variable"]].copy()

# Put counts back into X for DECODE, if you want DECODE to use counts
adata_hvg.X = adata_hvg.layers["counts"].copy()

if issparse(adata_hvg.X):
    adata_hvg.X = adata_hvg.X.toarray().astype("float32")
else:
    adata_hvg.X = adata_hvg.X.astype("float32")

print("After HVG selection:", adata_hvg)
print("Number of HVGs:", adata_hvg.n_vars)

adata_hvg.write_h5ad(out_path)

print(f"Saved: {out_path}")

Original: AnnData object with n_obs × n_vars = 75440 × 65258
    obs: 'pct_counts_in_top_500_genes', 'tissue', 'anatomical_position', 'non_mito_counts', 'non_mito_genes', 'doublet_score', 'pct_counts_in_top_200_genes', 'donor', 'batch', 'pct_counts_in_top_50_genes', 'cell_ontology_class', 'compartment', 'total_counts_mt', 'ethnicity', 'pct_counts_in_top_100_genes', 'total_counts', 'n_genes_by_counts', 'predicted_doublet', 'pct_counts_mt', 'MergedClusters'
    layers: 'counts', 'decontXcounts', 'raw_counts'
After HVG selection: AnnData object with n_obs × n_vars = 75440 × 20000
    obs: 'pct_counts_in_top_500_genes', 'tissue', 'anatomical_position', 'non_mito_counts', 'non_mito_genes', 'doublet_score', 'pct_counts_in_top_200_genes', 'donor', 'batch', 'pct_counts_in_top_50_genes', 'cell_ontology_class', 'compartment', 'total_counts_mt', 'ethnicity', 'pct_counts_in_top_100_genes', 'total_counts', 'n_genes_by_counts', 'predicted_doublet', 'pct_counts_mt', 'MergedClusters'
    var: 'highly_

In [8]:
import scanpy as sc
from scipy.sparse import issparse

in_path = "/gpfs/igmmfs01/datastore/wendy-lab/Alexis/CDA-Data/03_TSP_Matrix-Generation/HumanBrainAtlas-snRNA-V1/TSP-HBA-QC-Inner_matched_DECODE_1500each.h5ad"
out_path = "/gpfs/igmmfs01/datastore/wendy-lab/Alexis/CDA-Data/03_TSP_Matrix-Generation/HumanBrainAtlas-snRNA-V1/TSP-HBA-QC-Inner_matched_DECODE_1500each_20kHVG.h5ad"

adata = sc.read_h5ad(in_path)

print("Original:", adata)

# Make sure counts are used if available
if "counts" in adata.layers:
    adata.X = adata.layers["counts"].copy()

# Scanpy HVG expects normalized/log-transformed data for most flavors.
# Keep raw counts untouched in a layer.
adata.layers["counts"] = adata.X.copy()

# Normalize/log only for HVG selection
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

sc.pp.highly_variable_genes(
    adata,
    n_top_genes=20000,
    flavor="seurat",
)

adata_hvg = adata[:, adata.var["highly_variable"]].copy()

# Put counts back into X for DECODE, if you want DECODE to use counts
adata_hvg.X = adata_hvg.layers["counts"].copy()

if issparse(adata_hvg.X):
    adata_hvg.X = adata_hvg.X.toarray().astype("float32")
else:
    adata_hvg.X = adata_hvg.X.astype("float32")

print("After HVG selection:", adata_hvg)
print("Number of HVGs:", adata_hvg.n_vars)

adata_hvg.write_h5ad(out_path)

print(f"Saved: {out_path}")

Original: AnnData object with n_obs × n_vars = 83990 × 59189
    obs: 'pct_counts_in_top_500_genes', 'total_counts_mt', 'pct_counts_in_top_200_genes', 'predicted_doublet', 'total_counts', 'pct_counts_in_top_100_genes', 'method', 'cell_ontology_class', 'pct_counts_mt', 'donor', 'anatomical_position', 'n_genes_by_counts', 'compartment', 'batch', 'assay', 'sample_id', 'sex', 'cell_ontology_id', 'non_mito_genes', 'doublet_score', 'non_mito_counts', 'ethnicity', 'tissue', 'pct_counts_in_top_50_genes', 'MergedClusters'
    layers: 'counts'
After HVG selection: AnnData object with n_obs × n_vars = 83990 × 20000
    obs: 'pct_counts_in_top_500_genes', 'total_counts_mt', 'pct_counts_in_top_200_genes', 'predicted_doublet', 'total_counts', 'pct_counts_in_top_100_genes', 'method', 'cell_ontology_class', 'pct_counts_mt', 'donor', 'anatomical_position', 'n_genes_by_counts', 'compartment', 'batch', 'assay', 'sample_id', 'sex', 'cell_ontology_id', 'non_mito_genes', 'doublet_score', 'non_mito_counts', 